# Ask RAG

Один вопрос к RAG-пайплайну (**Gemini** / **Ollama**).

Перед запуском нужен FAISS-индекс (`scripts/build_index.py` или ячейка ниже).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)

ROOT: /home/romanrussia/Code_git/agent_lokis


In [2]:
# Build index once if missing
from src.config import settings
from src.rag.index import build_index

if not settings.index_dir.exists():
    print("Building FAISS index...")
    build_index()
    print("Done:", settings.index_dir)
else:
    print("Index exists:", settings.index_dir)

Index exists: /home/romanrussia/Code_git/agent_lokis/indexes/faiss_articles


In [3]:
from src.models.factory import get_llm
from src.rag.pipeline import RAGPipeline

BACKEND = "gemini"  # или "ollama"
QUESTION = "За сколько рабочих дней до поездки нужно подать заявку в TravelHub?"

llm = get_llm(BACKEND)
rag = RAGPipeline(llm=llm, top_k=3)

result = rag.answer(QUESTION)
print("Model:", result.model)
print("Answer:", result.answer)
print("Retrieved:", [h.title for h in result.hits])

/home/romanrussia/Code_git/agent_lokis/src/rag/index.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1785253485.356577   10851 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785253485.

Model: gemini-3.5-flash-lite
Answer: Не позднее чем за 5 рабочих дней до выезда.
Retrieved: ['Регламент оформления командировок и возмещения расходов', 'Политика удаленной работы и компенсации домашнего офиса', 'Руководство по онбордингу новых сотрудников']


In [4]:
# Контекст, который ушёл в LLM
print(result.context)

[1] Регламент оформления командировок и возмещения расходов (id=1)
С 1 января 2026 года в компании действует обновленный регламент командировок. Заявка оформляется в системе TravelHub не позднее чем за 5 рабочих дней до выезда и должна быть согласована руководителем подразделения и финансовым контролером. Базовый лимит на проживание по России установлен в размере 9000 рублей за ночь, по СНГ — 12000 рублей, по Европе — 160 евро. Лимит на суточные для сотрудников уровня Specialist и Senior составляет 2500 рублей в сутки, для руководителей групп — 3500 рублей

[2] Политика удаленной работы и компенсации домашнего офиса (id=8)
Сотрудники могут работать удаленно до 4 дней в неделю при сохранении присутствия в офисе в командный день, установленный руководителем. Компания компенсирует расходы на домашний офис в размере до 4000 рублей в месяц при предоставлении чеков через портал PeopleDesk до 5 числа следующего месяца. Компенсации подлежат интернет, рабочее кресло, клавиатура, гарнитура и вне